# Standalone Decision Tree Control Notebook

This notebook is fully standalone and does not import from the repo. It only needs the SQLite dataset file.

Estimated training time on the current dataset:
- Full dataset, 5 CV folds + final fit: about 8-25 minutes
- `max_tickers=200`: often 2-6 minutes

This model is CPU-bound. Colab GPU does not materially speed it up.
The saved `.pkl` payload is still compatible with the pipeline's `tree` loader.


In [ ]:
# Install only what this notebook needs.
!pip -q install pandas numpy scikit-learn joblib


In [ ]:
from pathlib import Path

# Point this to your SQLite file in Colab or Drive.
DB_PATH = Path('/content/stock_prices_20y.db')

# Standalone outputs. These artifacts are still pipeline-compatible.
OUTPUT_DIR = Path('/content/model_outputs')
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'results'
CONFIGS_DIR = OUTPUT_DIR / 'configs'

for path in [MODELS_DIR, RESULTS_DIR, CONFIGS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing SQLite database: {DB_PATH}')

print('DB_PATH    =', DB_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)


In [ ]:
import sqlite3
import time
from datetime import datetime

import numpy as np
import pandas as pd

with sqlite3.connect(DB_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')


In [ ]:
import json
import pickle
import sqlite3
from dataclasses import asdict, dataclass
from typing import List, Tuple

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

@dataclass
class Config:
    db_path: str = str(DB_PATH)
    table_name: str = 'prices'
    n_splits: int = 5
    test_size: float = 0.2
    target_horizon: int = 1
    random_state: int = 42
    max_tickers: int = 0
    tree_max_depth: int = 6
    tree_min_samples_leaf: int = 100
    tree_min_samples_split: int = 500
    model_output_path: str = str(MODELS_DIR / f'{RUN_TAG}-decision_tree.pkl')
    config_output_path: str = str(CONFIGS_DIR / f'{RUN_TAG}-decision_tree_config.json')

cfg = Config()
print(cfg)

FEATURE_COLS = [
    'log_ret_1', 'log_ret_3', 'log_ret_5', 'log_ret_10', 'log_ret_20', 'log_ret_60',
    'hl_range', 'oc_change',
    'close_vs_ma_5', 'close_vs_ma_10', 'close_vs_ma_20', 'close_vs_ma_50', 'close_vs_ma_200',
    'ma_5_vs_20', 'ma_20_vs_50',
    'rolling_vol_5', 'rolling_vol_10', 'rolling_vol_20', 'vol_of_vol_20',
    'log_vol_chg_1', 'log_vol_chg_5', 'vol_vs_ma_5', 'vol_vs_ma_20',
    'rsi_14', 'rsi_7', 'macd_norm', 'macd_signal_norm',
    'bb_pos', 'bb_width', 'atr_14_norm', 'skew_20', 'kurt_20',
]


def load_prices_from_sqlite(db_path: str, table_name: str) -> pd.DataFrame:
    conn = sqlite3.connect(db_path)
    query = f'''        SELECT ticker, date, open, high, low, close, adj_close, volume
        FROM {table_name}
        ORDER BY ticker, date
    '''
    df = pd.read_sql_query(query, conn)
    conn.close()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    for c in ['open', 'high', 'low', 'close', 'adj_close']:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0).astype('float64')
    return df


def _compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.clip(lower=0).ewm(alpha=1.0 / period, min_periods=period).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1.0 / period, min_periods=period).mean()
    return (100.0 - 100.0 / (1.0 + gain / (loss + 1e-8))) / 100.0 - 0.5


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = []
    for _, g in df.groupby('ticker', sort=False):
        g = g.sort_values('date').copy()
        close = g['close']
        high = g['high']
        low = g['low']
        open_ = g['open']
        volume = g['volume'].clip(lower=1.0)

        log_c = np.log(close)
        g['log_ret_1'] = log_c.diff(1)
        g['log_ret_3'] = log_c.diff(3)
        g['log_ret_5'] = log_c.diff(5)
        g['log_ret_10'] = log_c.diff(10)
        g['log_ret_20'] = log_c.diff(20)
        g['log_ret_60'] = log_c.diff(60)
        g['hl_range'] = (high - low) / close
        g['oc_change'] = (close - open_) / open_.clip(lower=1e-6)

        for d in [5, 10, 20, 50, 200]:
            ma = close.rolling(d).mean()
            g[f'close_vs_ma_{d}'] = close / ma.clip(lower=1e-6) - 1.0

        g['ma_5_vs_20'] = close.rolling(5).mean() / close.rolling(20).mean().clip(lower=1e-6) - 1.0
        g['ma_20_vs_50'] = close.rolling(20).mean() / close.rolling(50).mean().clip(lower=1e-6) - 1.0

        for d in [5, 10, 20]:
            g[f'rolling_vol_{d}'] = g['log_ret_1'].rolling(d).std()
        g['vol_of_vol_20'] = g['rolling_vol_5'].rolling(20).std()

        log_vol = np.log1p(volume)
        g['log_vol_chg_1'] = log_vol.diff(1)
        g['log_vol_chg_5'] = log_vol.diff(5)
        for d in [5, 20]:
            g[f'vol_vs_ma_{d}'] = volume / volume.rolling(d).mean().clip(lower=1.0) - 1.0

        g['rsi_14'] = _compute_rsi(close, 14)
        g['rsi_7'] = _compute_rsi(close, 7)

        ema12 = close.ewm(span=12, min_periods=12).mean()
        ema26 = close.ewm(span=26, min_periods=26).mean()
        macd = ema12 - ema26
        g['macd_norm'] = macd / close.clip(lower=1e-6)
        g['macd_signal_norm'] = macd.ewm(span=9, min_periods=9).mean() / close.clip(lower=1e-6)

        bb_mean = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        g['bb_pos'] = (close - (bb_mean - 2 * bb_std)) / (4 * bb_std).clip(lower=1e-6) - 0.5
        g['bb_width'] = 4 * bb_std / bb_mean.clip(lower=1e-6)

        prev_c = close.shift(1)
        tr = pd.concat([high - low, (high - prev_c).abs(), (low - prev_c).abs()], axis=1).max(axis=1)
        g['atr_14_norm'] = tr.ewm(alpha=1.0 / 14, min_periods=14).mean() / close.clip(lower=1e-6)

        g['skew_20'] = g['log_ret_1'].rolling(20).skew()
        g['kurt_20'] = g['log_ret_1'].rolling(20).kurt()
        g['target'] = np.log(close.shift(-1) / close)
        out.append(g)

    result = pd.concat(out, ignore_index=True)
    return result.replace([np.inf, -np.inf], np.nan)


def split_train_test_per_ticker(df: pd.DataFrame, test_size: float, embargo: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_parts, test_parts = [], []
    for _, grp in df.groupby('ticker', sort=False):
        grp = grp.sort_values('date').reset_index(drop=True)
        n = len(grp)
        split_idx = int(n * (1 - test_size))
        train_end = split_idx - embargo
        if split_idx <= 0 or split_idx >= n or train_end <= 0:
            continue
        train_parts.append(grp.iloc[:train_end].copy())
        test_parts.append(grp.iloc[split_idx:].copy())
    if not train_parts or not test_parts:
        raise ValueError('Empty train/test split.')
    return pd.concat(train_parts, ignore_index=True), pd.concat(test_parts, ignore_index=True)


def build_grouped_time_folds(df: pd.DataFrame, n_splits: int, embargo: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    df = df.sort_values(['ticker', 'date']).reset_index(drop=True)
    folds = []
    for fold_num in range(n_splits):
        tr_idx, va_idx = [], []
        for _, grp in df.groupby('ticker', sort=False):
            idx = grp.index.to_numpy()
            n = len(idx)
            if n < (n_splits + 1) * max(embargo, 1) + 5:
                continue
            boundaries = np.linspace(0, n, n_splits + 2, dtype=int)
            va_start = boundaries[fold_num + 1]
            va_end = boundaries[fold_num + 2]
            purge_end = va_start - embargo
            if purge_end <= 0 or va_end <= va_start:
                continue
            tr_idx.extend(idx[:purge_end].tolist())
            va_idx.extend(idx[va_start:va_end].tolist())
        if tr_idx and va_idx:
            folds.append((np.array(tr_idx), np.array(va_idx)))
    if not folds:
        raise ValueError('No CV folds built.')
    return folds


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) < 2:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'directional_accuracy': np.nan}
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'directional_accuracy': float(np.mean(np.sign(y_true) == np.sign(y_pred))),
    }


In [ ]:
start_time = time.perf_counter()

print('Loading data...')
raw_df = load_prices_from_sqlite(cfg.db_path, cfg.table_name)
all_tickers = list(raw_df['ticker'].unique())
if cfg.max_tickers > 0 and len(all_tickers) > cfg.max_tickers:
    rng = np.random.default_rng(cfg.random_state)
    all_tickers = rng.choice(all_tickers, cfg.max_tickers, replace=False).tolist()
    raw_df = raw_df[raw_df['ticker'].isin(all_tickers)].reset_index(drop=True)
    print(f'Subsampled to {cfg.max_tickers} tickers')
else:
    print(f'Using all {len(all_tickers)} tickers')

print('Computing features...')
feat_df = make_features(raw_df)
train_df, test_df = split_train_test_per_ticker(feat_df, cfg.test_size, cfg.target_horizon)
feature_cols = [c for c in FEATURE_COLS if c in train_df.columns]
train_clean = train_df[feature_cols + ['target', 'ticker', 'date', 'close']].dropna().copy()
test_clean = test_df[feature_cols + ['target', 'ticker', 'date', 'close']].dropna().copy()

X_train = train_clean[feature_cols].to_numpy(dtype=np.float32)
y_train = train_clean['target'].to_numpy(dtype=np.float32)
X_test = test_clean[feature_cols].to_numpy(dtype=np.float32)
y_test = test_clean['target'].to_numpy(dtype=np.float32)

print(f'Train rows: {len(X_train):,} | Test rows: {len(X_test):,}')
print('Building CV folds...')
folds = build_grouped_time_folds(train_clean, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

cv_results = []
for fold_i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    model = DecisionTreeRegressor(
        max_depth=cfg.tree_max_depth,
        min_samples_leaf=cfg.tree_min_samples_leaf,
        min_samples_split=cfg.tree_min_samples_split,
        random_state=cfg.random_state,
    )
    model.fit(X_train[tr_idx], y_train[tr_idx])
    metrics = regression_metrics(y_train[va_idx], model.predict(X_train[va_idx]))
    cv_results.append(metrics)
    print(metrics)

cv_df = pd.DataFrame(cv_results)
print('\nCV mean metrics')
print(cv_df.mean(numeric_only=True).to_string())

print('\nTraining final model...')
final_model = DecisionTreeRegressor(
    max_depth=cfg.tree_max_depth,
    min_samples_leaf=cfg.tree_min_samples_leaf,
    min_samples_split=cfg.tree_min_samples_split,
    random_state=cfg.random_state,
)
final_model.fit(X_train, y_train)
y_test_pred = final_model.predict(X_test)
test_metrics = regression_metrics(y_test, y_test_pred)
print('Held-out test metrics:', test_metrics)

payload = {
    'model': final_model,
    'feature_cols': feature_cols,
    'config': asdict(cfg),
    'cv_results': cv_results,
    'test_metrics': test_metrics,
}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
with open(cfg.config_output_path, 'w') as f:
    json.dump(asdict(cfg), f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-decision_tree_predictions.csv'
out_df = test_clean[['ticker', 'date', 'close']].copy()
out_df['y_true'] = y_test
out_df['y_pred'] = y_test_pred
out_df.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
